# Bitcoin Daily - Exploratory Data Analysis

This notebook explores the Bitcoin Daily price dataset.

**Dataset:** `btc-daily.csv`

**Description:** Daily Bitcoin price data (Spanish format) including last price, open, high, low, volume, and percentage change from 2013 to present.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

DATA_PATH = Path('../../data/raw/btc-daily.csv')

## 1. Load and Clean Data

In [ ]:
# Load dataset
df = pd.read_csv(DATA_PATH)

# Display first rows
print("First 5 rows (raw):")
display(df.head())

# Clean column names (Spanish to English)
df.columns = ['Date', 'Last', 'Open', 'High', 'Low', 'Vol.', 'Change %']

# Parse date (format: DD.MM.YYYY)
df['Date'] = pd.to_datetime(df['Date'], format='%d.%m.%Y')

# Clean numeric columns (Spanish format: comma as decimal separator, dot as thousands)
numeric_cols = ['Last', 'Open', 'High', 'Low']
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.replace('.', '').str.replace(',', '.').astype(float)

# Clean Volume column (handles K, M, B suffixes)
def clean_volume(v):
    if pd.isna(v) or v == '' or v == '-':
        return np.nan
    v = str(v).replace('.', '').replace(',', '.').upper()
    if 'K' in v:
        return float(v.replace('K', '')) * 1e3
    if 'M' in v:
        return float(v.replace('M', '')) * 1e6
    if 'B' in v:
        return float(v.replace('B', '')) * 1e9
    try:
        return float(v)
    except:
        return np.nan

if 'Vol.' in df.columns:
    df['Vol.'] = df['Vol.'].apply(clean_volume)

# Clean percentage column
df['Change %'] = df['Change %'].str.replace('%', '').str.replace(',', '.').astype(float)

# Sort by date
df = df.sort_values('Date').reset_index(drop=True)

print("\nCleaned dataset:")
display(df.head())
print("\nData types:")
print(df.dtypes)

## 2. Data Overview

In [ ]:
print("Dataset Shape:", df.shape)
print("\nDate range:", df['Date'].min(), "to", df['Date'].max())
print("Total trading days:", len(df))
print("\nDataset Info:")
df.info()

## 3. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Count': missing, 'Percentage': missing_pct})
print("Missing Values Summary:")
display(missing_df[missing_df['Missing Count'] > 0])
if missing.sum() == 0:
    print("\n✓ No missing values found!")

## 4. Descriptive Statistics

In [ ]:
print("Descriptive Statistics:")
display(df[['Last', 'Open', 'High', 'Low', 'Vol.', 'Change %']].describe())

## 5. Price Time Series

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['Date'], df['Last'], linewidth=1, alpha=0.8)
ax.set_title('Bitcoin Price - Closing Price Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price (USD)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Moving averages
df['MA_50'] = df['Last'].rolling(window=50).mean()
df['MA_200'] = df['Last'].rolling(window=200).mean()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['Date'], df['Last'], linewidth=1, alpha=0.6, label='Price')
ax.plot(df['Date'], df['MA_50'], linewidth=2, label='50-day MA', color='orange')
ax.plot(df['Date'], df['MA_200'], linewidth=2, label='200-day MA', color='red')
ax.set_title('Bitcoin with Moving Averages', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price (USD)', fontsize=12)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Log Scale Analysis

In [ ]:
# Bitcoin price in log scale (better for exponential growth)
fig, ax = plt.subplots(figsize=(16, 6))
ax.semilogy(df['Date'], df['Last'], linewidth=1, alpha=0.8)
ax.set_title('Bitcoin Price - Log Scale', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Price (USD) - Log Scale', fontsize=12)
ax.grid(True, alpha=0.3, which='both')
plt.tight_layout()
plt.show()

## 7. Returns Analysis

In [ ]:
df['Daily_Return'] = df['Last'].pct_change() * 100

fig, axes = plt.subplots(2, 1, figsize=(16, 10))

axes[0].plot(df['Date'], df['Daily_Return'], linewidth=0.5, alpha=0.7)
axes[0].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[0].set_title('Daily Returns Over Time', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Date')
axes[0].set_ylabel('Daily Return (%)')
axes[0].grid(True, alpha=0.3)

axes[1].hist(df['Daily_Return'].dropna(), bins=100, edgecolor='black', alpha=0.7)
axes[1].axvline(df['Daily_Return'].mean(), color='red', linestyle='--', label=f'Mean: {df["Daily_Return"].mean():.4f}%')
axes[1].set_title('Distribution of Daily Returns', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Daily Return (%)')
axes[1].set_ylabel('Frequency')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Volatility Analysis

In [ ]:
df['Volatility_30d'] = df['Daily_Return'].rolling(window=30).std()

fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(df['Date'], df['Volatility_30d'], linewidth=1, color='purple')
ax.set_title('30-Day Rolling Volatility', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Volatility (Std Dev of Returns)', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Volume Analysis

In [ ]:
# Plot trading volume
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

# Price
axes[0].plot(df['Date'], df['Last'], linewidth=1)
axes[0].set_title('Bitcoin Price', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Price (USD)')
axes[0].grid(True, alpha=0.3)

# Volume
axes[1].bar(df['Date'], df['Vol.'], width=1, alpha=0.7)
axes[1].set_title('Trading Volume', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Volume')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Yearly Performance

In [ ]:
# Extract year
df['Year'] = df['Date'].dt.year

# Yearly statistics
yearly_stats = df.groupby('Year')['Last'].agg(['mean', 'std', 'min', 'max', 'count'])
print("Yearly Statistics:")
display(yearly_stats)

# Plot yearly average
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(yearly_stats.index, yearly_stats['mean'], alpha=0.7, edgecolor='black')
ax.set_title('Average Bitcoin Price by Year', fontsize=14, fontweight='bold')
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Average Price (USD)', fontsize=12)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 11. Bull/Bear Markets

In [ ]:
# Identify major peaks and troughs
top_10_prices = df.nlargest(10, 'Last')[['Date', 'Last']]
print("Top 10 Highest Prices:")
display(top_10_prices)

bottom_10_prices = df.nsmallest(10, 'Last')[['Date', 'Last']]
print("\nTop 10 Lowest Prices:")
display(bottom_10_prices)

## 12. Correlation Analysis

In [ ]:
# Correlation matrix
corr_cols = ['Last', 'Open', 'High', 'Low', 'Vol.', 'Change %', 'Daily_Return']
corr_matrix = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8}, ax=ax)
ax.set_title('Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 13. Summary

In [ ]:
print("=" * 60)
print("KEY FINDINGS - BITCOIN DAILY")
print("=" * 60)
print(f"\n1. Dataset Coverage:")
print(f"   - Start: {df['Date'].min().strftime('%Y-%m-%d')}")
print(f"   - End: {df['Date'].max().strftime('%Y-%m-%d')}")
print(f"   - Days: {len(df):,}")
print(f"\n2. Price Statistics:")
print(f"   - Current: ${df['Last'].iloc[-1]:,.2f}")
print(f"   - All-time High: ${df['Last'].max():,.2f} ({df.loc[df['Last'].idxmax(), 'Date'].strftime('%Y-%m-%d')})")
print(f"   - All-time Low: ${df['Last'].min():,.2f} ({df.loc[df['Last'].idxmin(), 'Date'].strftime('%Y-%m-%d')})")
print(f"   - Average: ${df['Last'].mean():,.2f}")
print(f"\n3. Returns:")
print(f"   - Avg Daily Return: {df['Daily_Return'].mean():.4f}%")
print(f"   - Volatility: {df['Daily_Return'].std():.4f}%")
print(f"   - Best Day: {df['Daily_Return'].max():.2f}% ({df.loc[df['Daily_Return'].idxmax(), 'Date'].strftime('%Y-%m-%d')})")
print(f"   - Worst Day: {df['Daily_Return'].min():.2f}% ({df.loc[df['Daily_Return'].idxmin(), 'Date'].strftime('%Y-%m-%d')})")
print(f"\n4. Growth:")
total_return = ((df['Last'].iloc[-1] / df['Last'].iloc[0]) - 1) * 100
print(f"   - Total Return: {total_return:,.2f}%")
print("\n" + "=" * 60)